ZINGMP3

In [14]:
import requests
import pandas as pd
from collections import defaultdict

def fetch_artist_songs(artist_name):
    url = f"http://localhost:5000/api/artistsongs?name={artist_name}"
    response = requests.get(url)
    
    if response.status_code != 200:
        print("Error fetching data:", response.status_code)
        return None

    data = response.json()
    songs = data.get("songs", [])

    records = []
    album_tracks = defaultdict(set)

    for song in songs:
        album_link = song.get("albumLink", "")
        album_name = song.get("album") or song.get("title", "N/A")
        tracklist = song.get("tracklist") or [{"title": song.get("title", "N/A"), "link": song.get("link", "N/A")}]

        # Lấy album_type trực tiếp từ API (nếu có trong song data)
        album_type = song.get("albumType", "Unknown")
        
        # Lấy album_id từ song.albumId (giả sử dữ liệu albumId có sẵn)
        album_id = song.get("albumId", "Unknown")

        for track in tracklist:
            track_title = track.get("title", "N/A")
            album_tracks[album_id].add(track_title)

            records.append({
                "album_name": album_name,
                "tracklist": track_title,
                "release_date": song.get("releaseDate", "Unknown"),
                "provided_by": song.get("providedBy", "Unknown"),
                "featured_artists": song.get("featuredArtists", "Unknown"),
                "album_owner": song.get("albumOwner", "Unknown"),
                "album_id": album_id,
                "ZingMP3": track.get("link", "N/A"),
                "album_type": album_type
            })

    # Tạo DataFrame và xử lý dữ liệu
    df = pd.DataFrame(records)
    df.drop_duplicates(subset=["album_id", "tracklist"], inplace=True)
    df.sort_values(by=["album_name", "tracklist"], ascending=[True, True], inplace=True)

    return df

artist_name = 'Nghiem-Vu-Hoang-Long'
df = fetch_artist_songs(artist_name)
output_file = f'{artist_name}_songZingMP3.xlsx'
df.to_excel(output_file, index=False)
df.head()

,album_name,tracklist,release_date,provided_by,featured_artists,album_owner,album_id,ZingMP3,album_type
42,99%,00,02/03/2023,The Orchard,MCK,MCK,6B8AWD98,https://zingmp3.vn/bai-hat/00-MCK/Z66CUCCA.html,Regular
49,99%,50/50,02/03/2023,The Orchard,MCK,MCK,6B8AWD98,https://zingmp3.vn/bai-hat/50-50-MCK/Z66CUCEW....,Regular
57,99%,99,02/03/2023,The Orchard,MCK,MCK,6B8AWD98,https://zingmp3.vn/bai-hat/99-MCK/Z66CUCF7.html,Regular
35,99%,Ai Mới Là Kẻ Xấu Xa,02/03/2023,The Orchard,MCK,MCK,6B8AWD98,https://zingmp3.vn/bai-hat/Ai-Moi-La-Ke-Xau-Xa...,Regular
23,99%,Anh Đã Ổn Hơn,02/03/2023,The Orchard,MCK,MCK,6B8AWD98,https://zingmp3.vn/bai-hat/Anh-Da-On-Hon-MCK/Z...,Regular


SPOTIFY



In [10]:
import spotipy
import pandas as pd
from spotipy.oauth2 import SpotifyClientCredentials
import re

# Hàm lấy tất cả bài hát của nghệ sĩ
def get_artist_tracks_all(artist_name):
    client_id = "c7e1fe3ffe674920a01f9b016e6ae5df"
    client_secret = "215c6808dea74656b3629b306182ac4b"
    sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(client_id=client_id, client_secret=client_secret))

    # 🔎 Tìm nghệ sĩ theo tên
    results = sp.search(q=artist_name, type='artist', limit=1)
    if not results['artists']['items']:
        print(f"Không tìm thấy nghệ sĩ với tên '{artist_name}'.")
        return pd.DataFrame()

    artist_id = results['artists']['items'][0]['id']

    track_data = []
    seen_tracks = set()

    offset = 0
    while True:
        albums = sp.artist_albums(artist_id, album_type='album,single,compilation', limit=50, offset=offset)
        if not albums['items']:
            break

        for album in albums['items']:
            album_id = album['id']
            album_name = album['name']
            album_owner = album['artists'][0]['name']

            # Lấy thông tin album chi tiết
            album_info = sp.album(album_id)

            # ✅ Sửa chỗ lỗi ở đây
            album_date_raw = album_info.get('release_date', 'Unknown')
            try:
                album_release_date = pd.to_datetime(album_date_raw, errors='coerce').strftime('%d/%m/%Y')
            except:
                album_release_date = album_date_raw

            album_type = album_info.get('album_type', 'Unknown').capitalize()

            # Ưu tiên lấy thông tin nhà phát hành từ copyrights
            copyrights = album_info.get('copyrights', [])
            provider = 'Unknown'
            for c in copyrights:
                if c.get('type') in ('P', 'C') and c.get('text'):
                    provider = c['text']
                    break
            if provider == 'Unknown':
                provider = album_info.get('label', 'Unknown')

            # Lấy tất cả track trong album
            tracks = album_info.get('tracks', {}).get('items', [])
            track_count = len(tracks)

            # Phân loại album dựa trên số lượng bài hát
            if track_count <= 3:
                album_class = "Single"
            elif 4 <= track_count <= 6:
                album_class = "EP"
            else:
                album_class = "Regular"

            # Đưa thông tin bài hát vào track_data
            for track in tracks:
                track_id = track['id']
                if track_id in seen_tracks:
                    continue
                seen_tracks.add(track_id)

                track_title = track['name']
                link_spotify = track['external_urls']['spotify']
                featured_artists = [artist['name'] for artist in track['artists'] if artist['id'] != artist_id]
                featured_artists = ", ".join(featured_artists) if featured_artists else "None"

                track_data.append([
                    album_name, track_title, album_release_date, featured_artists,
                    album_owner, provider, link_spotify, album_class
                ])
        offset += 50

    # Lấy top track
    top_tracks = sp.artist_top_tracks(artist_id, country="US")['tracks']
    for track in top_tracks:
        track_id = track['id']
        if track_id in seen_tracks:
            continue
        seen_tracks.add(track_id)

        track_title = track['name']
        link_spotify = track['external_urls']['spotify']
        album = track['album']
        album_name = album['name']
        album_owner = album['artists'][0]['name']

        album_info = sp.album(album['id'])
        album_type = album_info.get('album_type', 'Unknown').capitalize()

        # ✅ Tương tự phần xử lý ngày
        album_date_raw = album_info.get('release_date', 'Unknown')
        try:
            album_release_date = pd.to_datetime(album_date_raw, errors='coerce').strftime('%d/%m/%Y')
        except:
            album_release_date = album_date_raw

        copyrights = album_info.get('copyrights', [])
        provider = 'Unknown'
        for c in copyrights:
            if c.get('type') in ('P', 'C') and c.get('text'):
                provider = c['text']
                break
        if provider == 'Unknown':
            provider = album_info.get('label', 'Unknown')

        featured_artists = [artist['name'] for artist in track['artists'] if artist['id'] != artist_id]
        featured_artists = ", ".join(featured_artists) if featured_artists else "None"

        track_data.append([
            album_name, track_title, album_release_date, featured_artists,
            album_owner, provider, link_spotify, album_type
        ])

    columns = ["album_name", "tracklist", "release_date", "featured_artists", "album_owner", "provided_by", "Link_Spotify", "album_type"]
    df = pd.DataFrame(track_data, columns=columns)
    df = df.sort_values(by=["album_name", "tracklist"], ascending=[True, True])
    return df


# Hàm xử lý provider
def extract_licensing_provider(provider_info):
    provider_info = re.sub(r"\(C\) \d{4}", "", provider_info)
    provider_info = re.sub(r"\d{4}", "", provider_info)
    provider_info = provider_info.replace("©", "").strip()

    match = re.search(r"exclusively licensed to (.+)", provider_info)
    return match.group(1).strip() if match else provider_info.strip()


# Chạy chương trình
if __name__ == "__main__":
    artist_name_Spotify = 'RPT MCK'
    df_tracks = get_artist_tracks_all(artist_name_Spotify)
    df_tracks['provided_by'] = df_tracks['provided_by'].apply(extract_licensing_provider)
    print(df_tracks.head())




   album_name            tracklist release_date featured_artists album_owner  \
0         99%                   00   02/03/2023             None     RPT MCK   
7         99%                50/50   02/03/2023             None     RPT MCK   
15        99%                   99   02/03/2023             None     RPT MCK   
12        99%  Ai Mới Là Kẻ Xấu Xa   02/03/2023             None     RPT MCK   
13        99%        Anh Đã Ổn Hơn   02/03/2023             None     RPT MCK   

   provided_by                                       Link_Spotify album_type  
0         CDSL  https://open.spotify.com/track/3xlhYIhZ7heAvoh...    Regular  
7         CDSL  https://open.spotify.com/track/33dIUFKBA7U5KHs...    Regular  
15        CDSL  https://open.spotify.com/track/4Mne52NZGUzdlPZ...    Regular  
12        CDSL  https://open.spotify.com/track/6GUGn0yUS6PvyYI...    Regular  
13        CDSL  https://open.spotify.com/track/3YctJXK6kznnWl6...    Regular  


In [11]:
df_tracks.head()

,album_name,tracklist,release_date,featured_artists,album_owner,provided_by,Link_Spotify,album_type
0,99%,00,02/03/2023,None,RPT MCK,CDSL,https://open.spotify.com/track/3xlhYIhZ7heAvoh...,Regular
7,99%,50/50,02/03/2023,None,RPT MCK,CDSL,https://open.spotify.com/track/33dIUFKBA7U5KHs...,Regular
15,99%,99,02/03/2023,None,RPT MCK,CDSL,https://open.spotify.com/track/4Mne52NZGUzdlPZ...,Regular
12,99%,Ai Mới Là Kẻ Xấu Xa,02/03/2023,None,RPT MCK,CDSL,https://open.spotify.com/track/6GUGn0yUS6PvyYI...,Regular
13,99%,Anh Đã Ổn Hơn,02/03/2023,None,RPT MCK,CDSL,https://open.spotify.com/track/3YctJXK6kznnWl6...,Regular


ZINGMP3+SPOTIFY

In [17]:
import pandas as pd
from thefuzz import process

# Chọn các cột cần thiết từ cả hai nguồn
df_tracks = df_tracks[
    [
        'album_name', 'tracklist', 'release_date', 'provided_by', 'Link_Spotify', 'album_type','album_owner','featured_artists'
    ]
]

df = df[
    [
        'album_id', 'album_name', 'tracklist', 'release_date', 'provided_by',
        'featured_artists', 'album_owner', 'ZingMP3', 'album_type'
    ]
]

# Tạo danh sách các tên album từ nguồn ZingMP3
album_names_zing = df['album_name'].tolist()

def find_best_match(album_name):
    result = process.extractOne(album_name, album_names_zing, score_cutoff=80)
    if result:  
        match, score = result  
        return match
    return album_name  

# Áp dụng fuzzy matching vào dữ liệu Spotify
df_tracks['album_name_matched'] = df_tracks['album_name'].apply(find_best_match)

# Gộp dữ liệu dựa trên album_name_matched, tracklist, album_type
df_merged = pd.merge(
    df_tracks, df, left_on=["album_name_matched", "tracklist", "album_type"], 
    right_on=["album_name", "tracklist", "album_type"], how="outer", suffixes=("_Spotify", "_ZingMP3")
)

# **Chọn tên album ưu tiên theo Spotify, nếu không có thì lấy từ ZingMP3**
df_merged["album_name_final"] = df_merged["album_name_Spotify"].combine_first(df_merged["album_name_ZingMP3"])

# Đổi tên cột
df_ = df_merged.rename(columns={
    "album_name_final": "album_name",
    "tracklist": "tracklist(danh sách bài hát)",
    "featured_artists_Spotify": "Song artist(nghệ sĩ tham gia bài hát)(Spotify)",
    "album_owner_Spotify": "Album artist (nghệ sĩ sở hữu album)*(Spotify)",
    "release_date_Spotify": "Ngày phát hành trên Spotify",
    "provided_by_ZingMP3": "Cung cấp bởi(ZingMP3)",
    "provided_by_Spotify": "Cung cấp bởi(Spotify)",
    "featured_artists_ZingMP3": "Song artist(nghệ sĩ tham gia bài hát)(ZingMP3)",
    "album_owner_ZingMP3": "Album artist (nghệ sĩ sở hữu album)*(ZingMP3)",
    "release_date_ZingMP3": "Ngày phát hành trên ZingMP3",
    "ZingMP3": "Link_ZingMP3",
    "Link_Spotify": "Spotify",
    "album_id": "Mã định danh album ZingMP3"
})

# Chọn các cột cần thiết
desired_columns = [
    "album_name",
    "Album artist (nghệ sĩ sở hữu album)*(Spotify)",  
    "Album artist (nghệ sĩ sở hữu album)*(ZingMP3)",
    "album_type",
    "tracklist(danh sách bài hát)",
    "Ngày phát hành trên Spotify",
    "Ngày phát hành trên ZingMP3",
    "Song artist(nghệ sĩ tham gia bài hát)(Spotify)",
    "Song artist(nghệ sĩ tham gia bài hát)(ZingMP3)",
    "Cung cấp bởi(ZingMP3)", 
    "Cung cấp bởi(Spotify)", 
    "Mã định danh album ZingMP3",
    "Link_ZingMP3",
    "Spotify"
]

df_ = df_[desired_columns]
# Xuất file Excel
df_.to_excel("song_ZingMP3+Spotify.xlsx", index=False)
df_.head()



,album_name,Album artist (nghệ sĩ sở hữu album)*(Spotify),Album artist (nghệ sĩ sở hữu album)*(ZingMP3),album_type,tracklist(danh sách bài hát),Ngày phát hành trên Spotify,Ngày phát hành trên ZingMP3,Song artist(nghệ sĩ tham gia bài hát)(Spotify),Song artist(nghệ sĩ tham gia bài hát)(ZingMP3),Cung cấp bởi(ZingMP3),Cung cấp bởi(Spotify),Mã định danh album ZingMP3,Link_ZingMP3,Spotify
0,99%,RPT MCK,MCK,Regular,00,02/03/2023,02/03/2023,None,MCK,The Orchard,CDSL,6B8AWD98,https://zingmp3.vn/bai-hat/00-MCK/Z66CUCCA.html,https://open.spotify.com/track/3xlhYIhZ7heAvoh...
1,99%,RPT MCK,MCK,Regular,50/50,02/03/2023,02/03/2023,None,MCK,The Orchard,CDSL,6B8AWD98,https://zingmp3.vn/bai-hat/50-50-MCK/Z66CUCEW....,https://open.spotify.com/track/33dIUFKBA7U5KHs...
2,99%,RPT MCK,MCK,Regular,99,02/03/2023,02/03/2023,None,MCK,The Orchard,CDSL,6B8AWD98,https://zingmp3.vn/bai-hat/99-MCK/Z66CUCF7.html,https://open.spotify.com/track/4Mne52NZGUzdlPZ...
3,99%,RPT MCK,MCK,Regular,Ai Mới Là Kẻ Xấu Xa,02/03/2023,02/03/2023,None,MCK,The Orchard,CDSL,6B8AWD98,https://zingmp3.vn/bai-hat/Ai-Moi-La-Ke-Xau-Xa...,https://open.spotify.com/track/6GUGn0yUS6PvyYI...
4,99%,RPT MCK,MCK,Regular,Anh Đã Ổn Hơn,02/03/2023,02/03/2023,None,MCK,The Orchard,CDSL,6B8AWD98,https://zingmp3.vn/bai-hat/Anh-Da-On-Hon-MCK/Z...,https://open.spotify.com/track/3YctJXK6kznnWl6...


APPLE MUSIC

In [18]:
import jwt  # Install with: pip install pyjwt
import time
import requests
import pandas as pd
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

TEAM_ID = '6A3CSCZ9M6'
KEY_ID = '2VLJH6R856'
PRIVATE_KEY_PATH = r'D:\AuthKey_2VLJH6R856.p8'

def generate_apple_music_token():
    with open(PRIVATE_KEY_PATH, "r") as key_file:
        private_key = key_file.read()
    
    payload = {
        "iss": TEAM_ID,
        "iat": int(time.time()),
        "exp": int(time.time()) + 3600,  
    }

    token = jwt.encode(payload, private_key, algorithm="ES256", headers={"alg": "ES256", "kid": KEY_ID})
    return token

APPLE_MUSIC_TOKEN = generate_apple_music_token()

In [19]:
def get_artist_albums(artist_id, storefront="us"):
    """ Lấy danh sách album của nghệ sĩ """
    url = f"https://api.music.apple.com/v1/catalog/{storefront}/artists/{artist_id}/albums"
    headers = {"Authorization": f"Bearer {APPLE_MUSIC_TOKEN}"}

    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        albums = response.json().get("data", [])
        return [
            {
                "album_id": album["id"],
                "album_name": album["attributes"]["name"],
                "release_date": album["attributes"]["releaseDate"],
                "medium": "single" if album["attributes"]["isSingle"] else "album",
                "genre": ", ".join(album["attributes"].get("genreNames", [])),
                "album_url": album["attributes"]["url"],
                "label": album["attributes"].get("recordLabel", "Unknown")
            }
            for album in albums
        ]
    return []

def get_album_tracks(album_id, storefront="us"):
    """ Lấy danh sách bài hát trong album """
    url = f"https://api.music.apple.com/v1/catalog/{storefront}/albums/{album_id}/tracks"
    headers = {"Authorization": f"Bearer {APPLE_MUSIC_TOKEN}"}

    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        tracks = response.json().get("data", [])
        return [
            {
                "album_id": album_id,
                "tracklist(danh sách bài hát)": track["attributes"]["name"],
                "track_url": track["attributes"]["url"],
                "featured_artists": ", ".join(
                    [artist["attributes"]["name"] for artist in track.get("relationships", {}).get("artists", {}).get("data", [])]
                ) if track.get("relationships", {}).get("artists") else track["attributes"].get("artistName", "None"),
            }
            for track in tracks
        ]
    return []

artist_ids = ["1533293457"]

# 🔹 Lấy thông tin album của nghệ sĩ
albums_data = []
for artist_id in artist_ids:
    albums_data.extend(get_artist_albums(artist_id))

df_albums = pd.DataFrame(albums_data)

# 🔹 Lấy thông tin bài hát của album đồng thời
tracks_data = []
with ThreadPoolExecutor() as executor:
    results = executor.map(lambda album: get_album_tracks(album["album_id"]), albums_data)

for album, track_list in zip(albums_data, results):
    for track in track_list:
        track.update({
            "album_name": album["album_name"],
            "release_date": album["release_date"],
            "status_code": "Normal",
            "class": "digital",
            "genre": album["genre"],
            "medium": album["medium"],
            "label": album["label"]

        })
        tracks_data.append(track)

df_tracks_apple = pd.DataFrame(tracks_data)
df_tracks_apple.head()

,album_id,tracklist(danh sách bài hát),track_url,featured_artists,album_name,release_date,status_code,class,genre,medium,label
0,1759395422,"Buồn Hay Vui (feat. RPT MCK, Obito, Ronboogz &...",https://music.apple.com/us/album/bu%E1%BB%93n-...,VSOUL,"Buồn Hay Vui (feat. RPT MCK, Obito, Ronboogz &...",2023-12-24,Normal,digital,"Hip-Hop/Rap, Music",single,12 trái lê
1,1611020804,Chìm Sâu (feat. Trung Trần),https://music.apple.com/us/album/ch%C3%ACm-s%C...,RPT MCK,Chìm Sâu (feat. Trung Trần) - Single,2022-02-24,Normal,digital,"Contemporary R&B, Music, R&B/Soul",single,CDSL
2,1670110613,00,https://music.apple.com/us/album/00/1670110613...,RPT MCK,99%,2023-03-02,Normal,digital,"Pop, Music",album,N0L4B3L
3,1670110613,Chìm Sâu (feat. Trung Trần),https://music.apple.com/us/album/ch%C3%ACm-s%C...,RPT MCK,99%,2023-03-02,Normal,digital,"Pop, Music",album,N0L4B3L
4,1670110613,Suit & Tie (feat. Hoàng Tôn),https://music.apple.com/us/album/suit-tie-feat...,RPT MCK,99%,2023-03-02,Normal,digital,"Pop, Music",album,N0L4B3L


ZINGMP3+SPOTIFY+APPLE


In [20]:
import pandas as pd
from fuzzywuzzy import fuzz, process

# Chuẩn hóa văn bản (Title Case)
def normalize_text(text):
    if isinstance(text, str):
        return text.strip().lower().title()
    return text

# Hàm fuzzy match để tìm match gần nhất
def fuzzy_match_single(value, choices):
    best_match = process.extractOne(value, choices, scorer=fuzz.token_sort_ratio)
    return best_match[0] if best_match else None

# Chuẩn hóa văn bản cho các cột tên album và bài hát
df_["album_name"] = df_["album_name"].apply(normalize_text)
df_tracks_apple["album_name"] = df_tracks_apple["album_name"].apply(normalize_text)
df_["tracklist(danh sách bài hát)"] = df_["tracklist(danh sách bài hát)"].apply(normalize_text)
df_tracks_apple["tracklist(danh sách bài hát)"] = df_tracks_apple["tracklist(danh sách bài hát)"].apply(normalize_text)

# Đảm bảo cột tracklist được đặt tên đúng
df_tracks_apple = df_tracks_apple.rename(columns={"tracklist": "tracklist(danh sách bài hát)"})

# Fuzzy match cho album_name giữa Apple Music và Zing/Spotify
df_tracks_apple['album_name_fuzzy'] = df_tracks_apple['album_name'].apply(
    lambda x: fuzzy_match_single(x, df_['album_name'])
)

# Merge theo tên album đã fuzzy match và tên bài hát
df_final = pd.merge(
    df_, 
    df_tracks_apple, 
    left_on=["album_name", "tracklist(danh sách bài hát)"], 
    right_on=["album_name_fuzzy", "tracklist(danh sách bài hát)"], 
    how="outer", 
    suffixes=("", "_Apple")
)

# Thêm cột "Cung cấp bởi(AppleMusic)" từ label nếu có
if "label" in df_tracks_apple.columns:
    df_final["Cung cấp bởi(AppleMusic)"] = df_final["label"]
else:
    df_final["Cung cấp bởi(AppleMusic)"] = ""
# Điền album_name còn thiếu từ AppleMusic (album_name_fuzzy)
df_final['album_name'] = df_final['album_name'].combine_first(df_final['album_name_fuzzy'])

# Xóa các cột không cần thiết
columns_to_drop = ["album_name_fuzzy", "album_id", "status_code", "class", "medium"]

df_final.drop(columns=[col for col in columns_to_drop if col in df_final.columns], inplace=True)


# Xóa trùng lặp
df_final.drop_duplicates(subset=["album_name", "tracklist(danh sách bài hát)"], keep="first", inplace=True)

# Đổi tên các cột theo yêu cầu
df_final = df_final.rename(columns={
    "album_owner_": "Nghệ sĩ sở hữu album", 
    "album_type": "album_type", 
    "genre": "genre", 
    "release_date": "Ngày phát hành trên AppleMusic", 
    "Song artist(nghệ sĩ tham gia bài hát)(Spotify)": "Song artist(nghệ sĩ tham gia bài hát)(Spotify)", 
    "Song artist(nghệ sĩ tham gia bài hát)(ZingMP3)": "Song artist(nghệ sĩ tham gia bài hát)(ZingMP3)", 
    "featured_artists": "Song artist(nghệ sĩ tham gia bài hát)(AppleMusic)", 
    "Cung cấp bởi(ZingMP3)": "Cung cấp bởi(ZingMP3)", 
    "Cung cấp bởi(Spotify)": "Cung cấp bởi(Spotify)", 
    "Link_ZingMP3": "ZingMP3", 
    "Spotify": "Spotify", 
    "track_url": "Apple Music"
})

# Chuyển định dạng ngày AppleMusic
df_final["Ngày phát hành trên AppleMusic"] = pd.to_datetime(
    df_final["Ngày phát hành trên AppleMusic"], errors='coerce'
).dt.strftime('%d/%m/%Y')

# Các cột cần xuất ra
final_columns = [
    'album_name', 'Nghệ sĩ sở hữu album', 'album_type', 'genre',
    'tracklist(danh sách bài hát)', 'Ngày phát hành trên Spotify',
    'Ngày phát hành trên ZingMP3', 'Ngày phát hành trên AppleMusic',
    'Song artist(nghệ sĩ tham gia bài hát)(Spotify)',
    'Song artist(nghệ sĩ tham gia bài hát)(ZingMP3)',
    'Song artist(nghệ sĩ tham gia bài hát)(AppleMusic)',
    'Cung cấp bởi(ZingMP3)', 'Cung cấp bởi(Spotify)', 'Cung cấp bởi(AppleMusic)',
    'Mã định danh ZingMP3',  
    'ZingMP3', 'Spotify', 'Apple Music'
]

# Lọc giữ lại các cột
df_final = df_final[[col for col in final_columns if col in df_final.columns]]

# Xuất file Excel
df_final.to_excel(f"{artist_name}_AlbumsZingMp3_Spot_Apple.xlsx", index=False)

# Xem trước dữ liệu
df_final.head()


c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


,album_name,album_type,genre,tracklist(danh sách bài hát),Ngày phát hành trên Spotify,Ngày phát hành trên ZingMP3,Ngày phát hành trên AppleMusic,Song artist(nghệ sĩ tham gia bài hát)(Spotify),Song artist(nghệ sĩ tham gia bài hát)(ZingMP3),Song artist(nghệ sĩ tham gia bài hát)(AppleMusic),Cung cấp bởi(ZingMP3),Cung cấp bởi(Spotify),Cung cấp bởi(AppleMusic),ZingMP3,Spotify,Apple Music
0,99%,Regular,"Pop, Music",00,02/03/2023,02/03/2023,02/03/2023,None,MCK,RPT MCK,The Orchard,CDSL,N0L4B3L,https://zingmp3.vn/bai-hat/00-MCK/Z66CUCCA.html,https://open.spotify.com/track/3xlhYIhZ7heAvoh...,https://music.apple.com/us/album/00/1670110613...
1,99%,Regular,"Pop, Music",50/50,02/03/2023,02/03/2023,02/03/2023,None,MCK,RPT MCK,The Orchard,CDSL,N0L4B3L,https://zingmp3.vn/bai-hat/50-50-MCK/Z66CUCEW....,https://open.spotify.com/track/33dIUFKBA7U5KHs...,https://music.apple.com/us/album/50-50/1670110...
2,99%,Regular,"Pop, Music",99,02/03/2023,02/03/2023,02/03/2023,None,MCK,RPT MCK,The Orchard,CDSL,N0L4B3L,https://zingmp3.vn/bai-hat/99-MCK/Z66CUCF7.html,https://open.spotify.com/track/4Mne52NZGUzdlPZ...,https://music.apple.com/us/album/99/1670110613...
3,99%,Regular,"Pop, Music",Ai Mới Là Kẻ Xấu Xa,02/03/2023,02/03/2023,02/03/2023,None,MCK,RPT MCK,The Orchard,CDSL,N0L4B3L,https://zingmp3.vn/bai-hat/Ai-Moi-La-Ke-Xau-Xa...,https://open.spotify.com/track/6GUGn0yUS6PvyYI...,https://music.apple.com/us/album/ai-m%E1%BB%9B...
4,99%,Regular,"Pop, Music",Anh Đã Ổn Hơn,02/03/2023,02/03/2023,02/03/2023,None,MCK,RPT MCK,The Orchard,CDSL,N0L4B3L,https://zingmp3.vn/bai-hat/Anh-Da-On-Hon-MCK/Z...,https://open.spotify.com/track/3YctJXK6kznnWl6...,https://music.apple.com/us/album/anh-%C4%91%C3...
